In [3]:
import torch
import triton
import triton.language as tl
import statistics

print(f"Triton version: {triton.__version__}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"PyTorch version: {torch.__version__}")

@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 256, "BLOCK_K": 64}, num_warps=8, num_stages=3),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 256, "BLOCK_K": 32}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128, "BLOCK_K": 32}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64,  "BLOCK_K": 32}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 128, "BLOCK_K": 32}, num_warps=4, num_stages=4),
        triton.Config({"BLOCK_M": 64,  "BLOCK_N": 64,  "BLOCK_K": 64}, num_warps=4, num_stages=4),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def matmul_kernel(
    A_ptr, B_ptr, C_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K, BLOCK_K):
        rk = k + tl.arange(0, BLOCK_K)
        a = tl.load(A_ptr + rm[:, None] * stride_am + rk[None, :] * stride_ak,
                    mask=(rm[:, None] < M) & (rk[None, :] < K), other=0.0)
        b = tl.load(B_ptr + rk[:, None] * stride_bk + rn[None, :] * stride_bn,
                    mask=(rk[:, None] < K) & (rn[None, :] < N), other=0.0)
        acc = tl.dot(a, b, acc)
    c = acc.to(tl.float16)
    tl.store(C_ptr + rm[:, None] * stride_cm + rn[None, :] * stride_cn,
             c, mask=(rm[:, None] < M) & (rn[None, :] < N))


def triton_matmul(A, B):
    M, K = A.shape
    _, N = B.shape
    C = torch.empty((M, N), dtype=torch.float16, device=A.device)
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]),
                         triton.cdiv(N, meta["BLOCK_N"]))
    matmul_kernel[grid](
        A, B, C, M, N, K,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
    )
    return C


def bench(fn, A, B, reps=200, warmup=50):
    for _ in range(warmup):
        fn(A, B)
    torch.cuda.synchronize()
    times = []
    for _ in range(reps):
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        fn(A, B)
        e.record()
        torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    return statistics.median(times)


# ── Correctness ──────────────────────────────────────────────────────────────
print("\n=== Correctness ===")
A = torch.randn(1024, 1024, dtype=torch.float16, device='cuda')
B = torch.randn(1024, 1024, dtype=torch.float16, device='cuda')
C_tri = triton_matmul(A, B)
C_ref = torch.matmul(A, B)
max_diff = (C_tri.float() - C_ref.float()).abs().max().item()
print(f"Max diff: {max_diff:.5f}")
print(f"Correct:  {max_diff < 0.1}")

# ── Multi-size benchmark ──────────────────────────────────────────────────────
print("\n=== Benchmark across matrix sizes (FP16) ===")
print(f"{'M':>6}  {'Triton ms':>10}  {'Triton GFLOPS':>14}  "
      f"{'cuBLAS ms':>10}  {'cuBLAS GFLOPS':>14}  {'Triton/cuBLAS':>13}")
print("-" * 80)

for M in [512, 1024, 2048, 4096, 8192]:
    A = torch.randn(M, M, dtype=torch.float16, device='cuda')
    B = torch.randn(M, M, dtype=torch.float16, device='cuda')

    tri_ms = bench(triton_matmul,                      A, B)
    cub_ms = bench(lambda a, b: torch.matmul(a, b),    A, B)

    flops   = 2 * M ** 3
    tri_gf  = flops / tri_ms  / 1e6
    cub_gf  = flops / cub_ms  / 1e6
    ratio   = tri_gf / cub_gf

    print(f"{M:>6}  {tri_ms:>10.3f}  {tri_gf:>14.1f}  "
          f"{cub_ms:>10.3f}  {cub_gf:>14.1f}  {ratio:>12.2f}x")

print(f"\nBest config (last size): {matmul_kernel.best_config}")

# ── Peak utilization ─────────────────────────────────────────────────────────
print("\n=== Peak utilization (T4 FP16 Tensor Core peak = 65,000 GFLOPS) ===")
A = torch.randn(4096, 4096, dtype=torch.float16, device='cuda')
B = torch.randn(4096, 4096, dtype=torch.float16, device='cuda')
tri_ms = bench(triton_matmul, A, B)
flops  = 2 * 4096 ** 3
tri_gf = flops / tri_ms / 1e6
print(f"Triton 4096x4096: {tri_gf:.1f} GFLOPS = {tri_gf/65000*100:.1f}% of T4 peak")

Triton version: 2.3.0
Device: Tesla T4
PyTorch version: 2.10.0+cu128

=== Correctness ===
Max diff: 0.00000
Correct:  True

=== Benchmark across matrix sizes (FP16) ===
     M   Triton ms   Triton GFLOPS   cuBLAS ms   cuBLAS GFLOPS  Triton/cuBLAS
--------------------------------------------------------------------------------
   512       0.140          1916.7       0.034          7955.1          0.24x
  1024       0.238          9008.5       0.072         29819.5          0.30x
  2048       0.994         17286.6       0.738         23275.4          0.74x
  4096       8.457         16252.1       5.727         24000.3          0.68x
  8192      66.896         16436.0      46.557         23616.5          0.70x

Best config (last size): BLOCK_M: 64, BLOCK_N: 256, BLOCK_K: 32, num_warps: 4, num_ctas: 1, num_stages: 4, enable_warp_specialization: False, enable_persistent: False

=== Peak utilization (T4 FP16 Tensor Core peak = 65,000 GFLOPS) ===
Triton 4096x4096: 15794.9 GFLOPS = 24.3% of T

In [1]:
pip install triton==2.3.0 --quiet